In [ ]:
import pandas as pd
import numpy as np
import os
import time
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch
import torchaudio
from torch.utils.data import Dataset
import pandas as pd

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/ravdess-emotional-speech-audio


In [ ]:
data=[]
audio_dir=f'{path}'
for actor in os.listdir(audio_dir):
    try:
      actor_files=os.path.join(audio_dir,actor)
      start_time=time.time()
      for file in os.listdir(actor_files):
          filename=actor+'_'+file
          file_path=os.path.join(audio_dir,actor,file)
          label=int(file.split('-')[2][1])-1
          data.append([file_path,label])
    except:
      continue
metadata=pd.DataFrame(data,columns=["file_path","label"])
metadata.to_csv("metadata.csv",sep=',',index=False)



In [ ]:
import sys
sys.executable

'/usr/bin/python3'

In [ ]:
import torch
import torchaudio
import pandas as pd
from tqdm import tqdm

bundle = torchaudio.pipelines.WAV2VEC2_BASE
model = bundle.get_model().eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

metadata = pd.read_csv("metadata.csv")

embeddings = []
labels = []

for i, row in tqdm(metadata.iterrows(), total=len(metadata)):
    path = row['file_path']
    label = row['label']

    waveform, sr = torchaudio.load(path)

    if sr != bundle.sample_rate:
        waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=bundle.sample_rate)(waveform)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    waveform = waveform.to(device)

    with torch.no_grad():
        features, _ = model.extract_features(waveform)
        embedding = features[-1].squeeze(0).mean(dim=0)  # [768]

    embeddings.append(embedding.cpu())
    labels.append(label)

# Save
torch.save((embeddings, labels), "precomputed_embeddings.pt")


Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960.pth
100%|██████████| 360M/360M [00:01<00:00, 260MB/s]
100%|██████████| 1440/1440 [01:05<00:00, 21.92it/s]


In [ ]:
class PrecomputedEmotionDataset(torch.utils.data.Dataset):
    def __init__(self, embedding_file):
        self.embeddings, self.labels = torch.load(embedding_file)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], torch.tensor(self.labels[idx], dtype=torch.long)


In [ ]:
import torch.nn as nn

class EmotionClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_classes=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
dataset = PrecomputedEmotionDataset("precomputed_embeddings.pt")
generator = torch.Generator().manual_seed(42)
train_set, test_set = torch.utils.data.random_split(
    dataset,
    [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))],
    generator=generator
)


train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32)


In [ ]:


# Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
model = EmotionClassifier().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

# Training loop
for epoch in range(500):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        print(f'Epoch: {epoch+1} Loss: {loss.item()}')

    print(f"Epoch {epoch+1}: Total Loss = {total_loss / len(train_loader):.4f}")


Streaming output truncated to the last 5000 lines.
Epoch: 365 Loss: 0.6110225915908813
Epoch: 365 Loss: 0.6269131898880005
Epoch: 365 Loss: 0.5741938352584839
Epoch: 365 Loss: 0.5802640318870544
Epoch 365: Total Loss = 0.6180
Epoch: 366 Loss: 0.6141954660415649
Epoch: 366 Loss: 0.6157327890396118
Epoch: 366 Loss: 0.6038829684257507
Epoch: 366 Loss: 0.5668959617614746
Epoch: 366 Loss: 0.6120818257331848
Epoch: 366 Loss: 0.6176562905311584
Epoch: 366 Loss: 0.5980784296989441
Epoch: 366 Loss: 0.5918710231781006
Epoch: 366 Loss: 0.6436346769332886
Epoch: 366 Loss: 0.6047512292861938
Epoch: 366 Loss: 0.6183161735534668
Epoch: 366 Loss: 0.6010905504226685
Epoch: 366 Loss: 0.6868796944618225
Epoch: 366 Loss: 0.6518869996070862
Epoch: 366 Loss: 0.6215832233428955
Epoch: 366 Loss: 0.6868468523025513
Epoch: 366 Loss: 0.5854918360710144
Epoch: 366 Loss: 0.6252319812774658
Epoch: 366 Loss: 0.6338100433349609
Epoch: 366 Loss: 0.619667112827301
Epoch: 366 Loss: 0.647217869758606
Epoch: 366 Loss: 0.5

In [ ]:
# from sklearn.metrics import accuracy_score,confusion_matrix

# model.eval()
# all_preds, all_labels = [], []

# with torch.no_grad():
#     for x, y in test_loader:
#         x, y = x.to(device), y.to(device)
#         out = model(x)
#         preds = torch.argmax(out, dim=1).cpu().numpy()
#         all_preds.extend(preds)
#         all_labels.extend(y.cpu().numpy())


# acc = accuracy_score(all_labels, all_preds)
# cf=confusion_matrix(all_labels, all_preds)
# print(f"Test Accuracy: {acc:.2f}")
# print(f"Confusion matrix:\n {cf}")


from sklearn.metrics import confusion_matrix
import numpy as np
import torch

model.eval()
all_top2_correct = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = model(x) 

        top2_preds = torch.topk(out, k=2, dim=1).indices  

    
        correct = top2_preds.eq(y.view(-1,1)).any(dim=1) 

        all_top2_correct.extend(correct.cpu().numpy())
        all_labels.extend(y.cpu().numpy())


top2_acc = np.mean(all_top2_correct)

print(f"Test Top-2 Accuracy: {top2_acc:.2f}")


Test Top-2 Accuracy: 0.92


In [ ]:
from sklearn.metrics import accuracy_score

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        preds = torch.argmax(out, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.cpu().numpy())


acc = accuracy_score(all_labels, all_preds)
cf=confusion_matrix(all_labels, all_preds)
print(f"Test Accuracy: {acc:.2f}")
print(f"Confusion matrix:\n {cf}")

Test Accuracy: 1.00
Confusion matrix:
 [[ 76   0   0   0   0   0   0   0]
 [  0 155   0   0   0   0   0   0]
 [  0   0 159   0   0   0   0   0]
 [  0   0   0 152   0   0   0   0]
 [  0   0   0   0 148   0   0   0]
 [  0   0   0   0   0 151   0   0]
 [  0   0   0   0   0   0 157   0]
 [  0   0   0   0   0   0   0 154]]


In [ ]:
torch.save(model.state_dict(), "model.pth")
# model.load_state_dict(torch.load("model.pth"))


In [ ]:
df=pd.read_csv('metadata.csv',sep=",")

In [ ]:
df['label'].value_counts()

In [ ]:
device